# 4. Data Cleaning and Missing Value Imputation

This notebook performs comprehensive data cleaning including:
- Missing value analysis and handling
- Outlier detection and removal
- Data validation and logical consistency checks
- Feature engineering for derived columns

## Setup and Data Loading

In [86]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

# Load data
data = pd.read_csv('cleaned_housing_data.csv')
data['zipcode'] = data['zipcode'].apply(lambda x: f"{x:05d}")

print(f"Initial dataset shape: {data.shape}")

# Remove duplicates
initial_count = len(data)
data = data.drop_duplicates()
duplicates_removed = initial_count - len(data)

if duplicates_removed > 0:
    print(f"Removed {duplicates_removed:,} duplicate records")
else:
    print("No duplicates found")

print(f"Dataset shape after duplicate removal: {data.shape}")
data.head()

Initial dataset shape: (366862, 28)
Removed 174 duplicate records
Dataset shape after duplicate removal: (366688, 28)


,status,propertyType,street,schools,zipcode,state,target,privatePool,propertyCategory,baths_num,...,cooling_energy_source,heating_type,heating_energy_source,parking_type,parking_spaces,has_garage,year_built,remodeled_year,lotsize_sqft,price_per_sqft
0,for sale,Single Family Home,240 Heather Ln,"[{'rating': ['4', '4', '7', 'NR', '4', '7', 'N...",28387,NC,418000.0,NaN,single_family,3.5,...,NaN,central,heat_pump,NaN,0,False,2019.0,NaN,NaN,144.0
1,for sale,single-family home,12911 E Heroy Ave,"[{'rating': ['4/10', 'None/10', '4/10'], 'data...",99216,WA,310000.0,NaN,single_family,3.0,...,NaN,NaN,NaN,NaN,0,False,2019.0,NaN,5828.0,159.0
2,for sale,single-family home,2005 Westridge Rd,"[{'rating': ['8/10', '4/10', '8/10'], 'data': ...",90049,CA,2895000.0,Yes,single_family,2.0,...,unknown,forced_air,unknown,attached_garage,1,True,1961.0,1967.0,8626.0,965.0
3,for sale,single-family home,4311 Livingston Ave,"[{'rating': ['9/10', '9/10', '10/10', '9/10'],...",75205,TX,2395000.0,NaN,single_family,8.0,...,unknown,forced_air,unknown,detached_garage,1,True,2006.0,2006.0,8220.0,371.0
4,for sale,lot/land,1524 Kiscoe St,"[{'rating': ['4/10', '5/10', '5/10'], 'data': ...",32908,FL,5000.0,NaN,land_lot,NaN,...,NaN,NaN,NaN,NaN,0,False,NaN,NaN,10019.0,NaN


## Missing Values Analysis

In [87]:
def analyze_missing_values(df):
    """Analyze missing values in the dataset"""
    missing_analysis = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
        'Data_Type': df.dtypes
    }).sort_values('Missing_Percentage', ascending=False)
    
    print(f"Dataset shape: {df.shape}")
    print(f"\nMissing Values Analysis:")
    print("=" * 60)
    print(missing_analysis.to_string(index=False))
    return missing_analysis

missing_analysis = analyze_missing_values(data)

Dataset shape: (366688, 28)

Missing Values Analysis:
               Column  Missing_Count  Missing_Percentage Data_Type
          privatePool         322370               87.91    object
       fireplace_type         267167               72.86    object
    fireplace_present         267167               72.86    object
      fireplace_count         267167               72.86   float64
   fireplace_location         267167               72.86    object
       remodeled_year         216231               58.97   float64
         parking_type         169309               46.17    object
          stories_num         148852               40.59   float64
cooling_energy_source         119740               32.65    object
         cooling_type         119740               32.65    object
            baths_num         110855               30.23   float64
             beds_num         110690               30.19   float64
heating_energy_source         105523               28.78    object
        

## Data Quality Filtering

Remove records with critical missing values that make them unusable for analysis.

In [88]:
# Remove records missing critical fields
print("Removing records with missing critical fields...")
initial_count = len(data)

# Drop records without status or property category
data = data.dropna(subset=['status', 'propertyCategory']).copy()

print(f"Removed {initial_count - len(data):,} records with missing status/propertyCategory")
print(f"Remaining records: {len(data):,}")

# Show final property category distribution
print(f"\nProperty category distribution:")
print(data['propertyCategory'].value_counts())

Removing records with missing critical fields...
Removed 70,971 records with missing status/propertyCategory
Remaining records: 295,717

Property category distribution:
propertyCategory
single_family      183417
condo_apartment     43961
land_lot            30826
townhouse           15587
multi_family        11335
co_op                3761
other                3618
mobile_home          3212
Name: count, dtype: int64


## Outlier Detection and Removal

Remove extreme outliers using z-score method on log-transformed data.

In [89]:
def remove_outliers_zscore(df, feature, log_scale=False, threshold=3):
    """Remove outliers using z-score method"""
    if log_scale:
        x = np.log(df[feature] + 1)
    else:
        x = df[feature]
    
    mu = x.mean()
    sigma = x.std()
    lower_bound = mu - threshold * sigma
    upper_bound = mu + threshold * sigma
    
    outliers_mask = (x < lower_bound) | (x > upper_bound)
    outliers_count = outliers_mask.sum()
    
    cleaned_df = df[~outliers_mask].copy()
    
    print(f"Removed {outliers_count:,} outliers from {feature}")
    print(f"Dataset size: {len(df):,} → {len(cleaned_df):,}")
    
    return cleaned_df

# Remove price outliers
data = remove_outliers_zscore(data, 'target', log_scale=True)

# Remove sqft outliers  
data = remove_outliers_zscore(data, 'sqft_num', log_scale=True)

# Remove lotsize outliers
data = remove_outliers_zscore(data, 'lotsize_sqft', log_scale=True)

# Remove extreme stories outliers
extreme_stories = (data['stories_num'] > 100)
if extreme_stories.sum() > 0:
    data = data[~extreme_stories].copy()
    print(f"Removed {extreme_stories.sum():,} properties with >100 stories")

Removed 4,825 outliers from target
Dataset size: 295,717 → 290,892
Removed 2,008 outliers from sqft_num
Dataset size: 290,892 → 288,884
Removed 3,770 outliers from lotsize_sqft
Dataset size: 288,884 → 285,114
Removed 1 properties with >100 stories


## Property-Specific Data Cleaning

Handle property-specific logical constraints and extreme values.

In [90]:
# Set land_lot properties to 0 for building characteristics
land_lot_mask = data['propertyCategory'] == 'land_lot'
land_lot_count = land_lot_mask.sum()

if land_lot_count > 0:
    print(f"Setting {land_lot_count:,} land_lot properties to 0 for beds/baths/sqft/stories")
    data.loc[land_lot_mask, ['beds_num', 'baths_num', 'sqft_num', 'stories_num']] = 0

# Remove properties with extreme beds/baths (>15)
extreme_beds = (data['beds_num'] > 15)
extreme_baths = (data['baths_num'] > 15)

total_extreme = extreme_beds.sum() + extreme_baths.sum()
if total_extreme > 0:
    data = data[~(extreme_beds | extreme_baths)].copy()
    print(f"Removed {total_extreme:,} properties with >15 beds or baths")

# Replace outliers in parking_spaces and fireplace_count with median by property type
def replace_outliers_with_median(df, column, threshold=10):
    """Replace extreme values with median by property type"""
    if column not in df.columns:
        return df
    
    outliers_mask = df[column] > threshold
    outliers_count = outliers_mask.sum()
    
    if outliers_count > 0:
        # Calculate median by property type
        median_by_type = df.groupby('propertyCategory')[column].median().to_dict()
        
        # Replace outliers with property type median
        for prop_type, median_val in median_by_type.items():
            type_outliers = outliers_mask & (df['propertyCategory'] == prop_type)
            if type_outliers.sum() > 0:
                df.loc[type_outliers, column] = median_val
        
        print(f"Replaced {outliers_count:,} outliers in {column} (>{threshold}) with median by property type")
    
    return df

# Replace parking_spaces outliers (>10 parking spaces)
data = replace_outliers_with_median(data, 'parking_spaces', threshold=10)

# Replace fireplace_count outliers (>5 fireplaces)  
data = replace_outliers_with_median(data, 'fireplace_count', threshold=5)

# Remove records with all key features missing
all_missing = (data['beds_num'].isna()) & (data['baths_num'].isna()) & (data['sqft_num'].isna())
if all_missing.sum() > 0:
    data = data[~all_missing].copy()
    print(f"Removed {all_missing.sum():,} records with all key features missing")

print(f"\nDataset after property-specific cleaning: {len(data):,} records")

Setting 25,711 land_lot properties to 0 for beds/baths/sqft/stories
Removed 9,334 properties with >15 beds or baths
Replaced 413 outliers in parking_spaces (>10) with median by property type
Replaced 51 outliers in fireplace_count (>5) with median by property type
Removed 2,482 records with all key features missing

Dataset after property-specific cleaning: 273,346 records


## Simple Feature Imputation

Fill missing values for simple categorical and boolean features.

In [91]:
# Fill simple categorical features
simple_fills = {
    'street': 'unknown',
    'privatePool': 'No',
    'fireplace_count': 0,
    'fireplace_present': 'No',
    'fireplace_location': 'unknown',
    'fireplace_type': 'unknown',
    'parking_type': 'unknown',
    'cooling_type': 'none',
    'cooling_energy_source': 'none',
    'heating_type': 'other',
    'heating_energy_source': 'other'
}

for column, fill_value in simple_fills.items():
    if column in data.columns:
        missing_count = data[column].isna().sum()
        if missing_count > 0:
            data[column] = data[column].fillna(fill_value)
            print(f"Filled {missing_count:,} missing values in {column} with '{fill_value}'")

Filled 2 missing values in street with 'unknown'
Filled 237,520 missing values in privatePool with 'No'
Filled 189,825 missing values in fireplace_count with '0'
Filled 189,825 missing values in fireplace_present with 'No'
Filled 189,825 missing values in fireplace_location with 'unknown'
Filled 189,825 missing values in fireplace_type with 'unknown'
Filled 115,772 missing values in parking_type with 'unknown'
Filled 84,916 missing values in cooling_type with 'none'
Filled 84,916 missing values in cooling_energy_source with 'none'
Filled 69,568 missing values in heating_type with 'other'
Filled 69,568 missing values in heating_energy_source with 'other'


## Date and Year Processing

Clean and process year-based columns with logical validation.

In [92]:
def process_year_columns(df):
    """Process year_built and remodeled_year with validation"""
    current_year = 2025
    
    # Clean invalid year_built values
    invalid_years = (df['year_built'] < 1700) & df['year_built'].notna()
    if invalid_years.sum() > 0:
        df = df[~invalid_years].copy()
        print(f"Removed {invalid_years.sum():,} records with invalid year_built")
    
    # Fill missing year_built with median by property type
    median_years = df.groupby('propertyCategory')['year_built'].median()
    df['missing_year_built'] = df['year_built'].isna().astype(int)
    
    missing_years = df['year_built'].isna().sum()
    if missing_years > 0:
        df['year_built'] = df['year_built'].fillna(df['propertyCategory'].map(median_years))
        print(f"Filled {missing_years:,} missing year_built values")
    
    # Fix remodeled_year inconsistencies
    invalid_remodel = (df['remodeled_year'] < df['year_built']) & df['remodeled_year'].notna()
    if invalid_remodel.sum() > 0:
        df.loc[invalid_remodel, 'remodeled_year'] = df.loc[invalid_remodel, 'year_built']
        print(f"Fixed {invalid_remodel.sum():,} invalid remodel years")
    
    # Create remodel indicator and fill missing remodeled_year
    df['was_remodeled'] = df['remodeled_year'].notna().astype(int)
    df['remodeled_year'] = df['remodeled_year'].fillna(df['year_built'])
    
    # Create derived features
    df['property_age'] = current_year - df['year_built']
    df['years_since_remodel'] = current_year - df['remodeled_year']
    df['years_build_to_remodelÏy'] = df['remodeled_year'] - df['year_built']
    
    print(f"Created derived year features")
    return df

data = process_year_columns(data)

Removed 1 records with invalid year_built
Filled 38,772 missing year_built values
Fixed 2,183 invalid remodel years
Created derived year features


## Price and Square Footage Recovery

Use price_per_sqft to recover missing values where possible.

In [93]:
# Calculate missing prices using price_per_sqft * sqft_num
price_calc_mask = (data['target'].isna()) & (data['sqft_num'].notna()) & (data['price_per_sqft'].notna())
if price_calc_mask.sum() > 0:
    data.loc[price_calc_mask, 'target'] = data.loc[price_calc_mask, 'sqft_num'] * data.loc[price_calc_mask, 'price_per_sqft']
    print(f"Calculated {price_calc_mask.sum():,} missing prices using price_per_sqft")

# Calculate missing sqft using target / price_per_sqft
sqft_calc_mask = (data['sqft_num'].isna()) & (data['target'].notna()) & (data['price_per_sqft'].notna()) & (data['price_per_sqft'] > 0)
if sqft_calc_mask.sum() > 0:
    data.loc[sqft_calc_mask, 'sqft_num'] = data.loc[sqft_calc_mask, 'target'] / data.loc[sqft_calc_mask, 'price_per_sqft']
    print(f"Calculated {sqft_calc_mask.sum():,} missing sqft using price_per_sqft")

# Drop remaining records with missing target
target_missing = data['target'].isna().sum()
if target_missing > 0:
    data = data.dropna(subset=['target']).copy()
    print(f"Dropped {target_missing:,} records with missing target price")

# Calculate new price per sqft and validate
# Only calculate for properties with sqft > 0 (exclude land lots)
data['ppsf'] = np.where(data['sqft_num'] > 0, data['target'] / data['sqft_num'], 0)

print(f"\nPrice per sqft statistics (before outlier removal):")
non_zero_ppsf = data[data['ppsf'] > 0]['ppsf']
print(f"Min: ${non_zero_ppsf.min():.0f}")
print(f"Max: ${non_zero_ppsf.max():.0f}")
print(f"Median: ${non_zero_ppsf.median():.0f}")
print(f"Mean: ${non_zero_ppsf.mean():.0f}")

# Remove ppsf outliers using z-score method (same as other features)
ppsf_for_analysis = data[data['ppsf'] > 0]['ppsf']  # Exclude land lots
if len(ppsf_for_analysis) > 0:
    log_ppsf = np.log(ppsf_for_analysis + 1)
    mu = log_ppsf.mean()
    sigma = log_ppsf.std()
    lower_bound = mu - 3 * sigma
    upper_bound = mu + 3 * sigma
    
    # Create outlier mask for properties with ppsf > 0
    ppsf_outliers_mask = data['ppsf'] > 0  # Start with non-zero ppsf
    log_ppsf_all = np.log(data[ppsf_outliers_mask]['ppsf'] + 1)
    ppsf_outliers = (log_ppsf_all < lower_bound) | (log_ppsf_all > upper_bound)
    
    # Apply outlier mask to full dataset
    outlier_indices = data[ppsf_outliers_mask].index[ppsf_outliers]
    extreme_ppsf_count = len(outlier_indices)
    
    if extreme_ppsf_count > 0:
        print(f"\nRemoving {extreme_ppsf_count:,} properties with extreme price per sqft (z-score > 3)")
        
        data = data.drop(outlier_indices).copy()
        print(f"Dataset size after removing extreme ppsf: {len(data):,}")

# Remove old price_per_sqft column (we have new ppsf)
data = data.drop(columns=['price_per_sqft'], errors='ignore')

print(f"\nFinal price per sqft statistics after z-score cleaning:")
final_ppsf = data[data['ppsf'] > 0]['ppsf']
if len(final_ppsf) > 0:
    print(f"Min: ${final_ppsf.min():.0f}")
    print(f"Max: ${final_ppsf.max():.0f}")
    print(f"Median: ${final_ppsf.median():.0f}")
    print(f"Records with ppsf > 0: {(data['ppsf'] > 0).sum():,}")

Calculated 4 missing prices using price_per_sqft
Calculated 390 missing sqft using price_per_sqft
Dropped 1,277 records with missing target price

Price per sqft statistics (before outlier removal):
Min: $1
Max: $134950
Median: $175
Mean: $262

Removing 2,322 properties with extreme price per sqft (z-score > 3)
Dataset size after removing extreme ppsf: 269,746

Final price per sqft statistics after z-score cleaning:
Min: $20
Max: $1813
Median: $175
Records with ppsf > 0: 240,401


## Smart Imputation for Property Features

Use contextual relationships to fill missing beds, baths, sqft, and stories.

In [94]:
def smart_impute_property_features(df):
    """Intelligently fill missing property features using contextual relationships"""
    
    # Fill sqft using beds + property type, then baths + property type
    missing_sqft = df['sqft_num'].isna().sum()
    if missing_sqft > 0:
        sqft_beds_lookup = df.groupby(['beds_num', 'propertyCategory'])['sqft_num'].median().to_dict()
        sqft_baths_lookup = df.groupby(['baths_num', 'propertyCategory'])['sqft_num'].median().to_dict()
        sqft_property_lookup = df.groupby('propertyCategory')['sqft_num'].median().to_dict()
        
        missing_mask = df['sqft_num'].isna()
        missing_data = df[missing_mask].copy()
        
        # Try beds + property type first
        missing_data['beds_key'] = list(zip(missing_data['beds_num'], missing_data['propertyCategory']))
        sqft_values = missing_data['beds_key'].map(sqft_beds_lookup)
        
        # Fallback to baths + property type
        fallback_mask = sqft_values.isna()
        if fallback_mask.sum() > 0:
            missing_data['baths_key'] = list(zip(missing_data['baths_num'], missing_data['propertyCategory']))
            sqft_values[fallback_mask] = missing_data.loc[fallback_mask, 'baths_key'].map(sqft_baths_lookup)
        
        # Final fallback to property type median
        final_fallback = sqft_values.isna()
        if final_fallback.sum() > 0:
            sqft_values[final_fallback] = missing_data.loc[final_fallback, 'propertyCategory'].map(sqft_property_lookup)
        
        df.loc[missing_mask, 'sqft_num'] = sqft_values
        print(f"Filled {missing_sqft:,} missing sqft values")
    
    # Fill baths using beds + property type, then sqft bins + property type
    missing_baths = df['baths_num'].isna().sum()
    if missing_baths > 0:
        baths_beds_lookup = df.groupby(['beds_num', 'propertyCategory'])['baths_num'].median().to_dict()
        
        # Create sqft bins for better grouping
        df['sqft_bin'] = pd.cut(df['sqft_num'], bins=10, precision=0)
        baths_sqft_lookup = df.groupby(['sqft_bin', 'propertyCategory'])['baths_num'].median().to_dict()
        baths_property_lookup = df.groupby('propertyCategory')['baths_num'].median().to_dict()
        
        missing_mask = df['baths_num'].isna()
        missing_data = df[missing_mask].copy()
        
        missing_data['beds_key'] = list(zip(missing_data['beds_num'], missing_data['propertyCategory']))
        missing_data['sqft_key'] = list(zip(missing_data['sqft_bin'], missing_data['propertyCategory']))
        
        baths_values = missing_data['beds_key'].map(baths_beds_lookup)
        
        fallback_mask = baths_values.isna()
        if fallback_mask.sum() > 0:
            baths_values[fallback_mask] = missing_data.loc[fallback_mask, 'sqft_key'].map(baths_sqft_lookup)
        
        final_fallback = baths_values.isna()
        if final_fallback.sum() > 0:
            baths_values[final_fallback] = missing_data.loc[final_fallback, 'propertyCategory'].map(baths_property_lookup)
        
        df.loc[missing_mask, 'baths_num'] = baths_values
        df = df.drop('sqft_bin', axis=1, errors='ignore')
        print(f"Filled {missing_baths:,} missing baths values")
    
    # Fill beds using baths + property type, then sqft bins + property type
    missing_beds = df['beds_num'].isna().sum()
    if missing_beds > 0:
        beds_baths_lookup = df.groupby(['baths_num', 'propertyCategory'])['beds_num'].median().to_dict()
        
        df['sqft_bin'] = pd.cut(df['sqft_num'], bins=10, precision=0)
        beds_sqft_lookup = df.groupby(['sqft_bin', 'propertyCategory'])['beds_num'].median().to_dict()
        beds_property_lookup = df.groupby('propertyCategory')['beds_num'].median().to_dict()
        
        missing_mask = df['beds_num'].isna()
        missing_data = df[missing_mask].copy()
        
        missing_data['baths_key'] = list(zip(missing_data['baths_num'], missing_data['propertyCategory']))
        missing_data['sqft_key'] = list(zip(missing_data['sqft_bin'], missing_data['propertyCategory']))
        
        beds_values = missing_data['baths_key'].map(beds_baths_lookup)
        
        fallback_mask = beds_values.isna()
        if fallback_mask.sum() > 0:
            beds_values[fallback_mask] = missing_data.loc[fallback_mask, 'sqft_key'].map(beds_sqft_lookup)
        
        final_fallback = beds_values.isna()
        if final_fallback.sum() > 0:
            beds_values[final_fallback] = missing_data.loc[final_fallback, 'propertyCategory'].map(beds_property_lookup)
        
        df.loc[missing_mask, 'beds_num'] = beds_values
        df = df.drop('sqft_bin', axis=1, errors='ignore')
        print(f"Filled {missing_beds:,} missing beds values")
    
    # Fill stories using property type median
    missing_stories = df['stories_num'].isna().sum()
    if missing_stories > 0:
        stories_lookup = df.groupby('propertyCategory')['stories_num'].median().to_dict()
        df['stories_num'] = df['stories_num'].fillna(df['propertyCategory'].map(stories_lookup))
        print(f"Filled {missing_stories:,} missing stories values")
    
    return df

data = smart_impute_property_features(data)

Filled 3,643 missing sqft values


/var/folders/6k/1096gq_54_dgk0_cjz4762g40000gn/T/ipykernel_20624/4266724295.py:39: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  baths_sqft_lookup = df.groupby(['sqft_bin', 'propertyCategory'])['baths_num'].median().to_dict()


Filled 45,845 missing baths values


/var/folders/6k/1096gq_54_dgk0_cjz4762g40000gn/T/ipykernel_20624/4266724295.py:68: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  beds_sqft_lookup = df.groupby(['sqft_bin', 'propertyCategory'])['beds_num'].median().to_dict()


Filled 35,248 missing beds values
Filled 73,636 missing stories values


## Lot Size Processing

Clean and fill lot size values with logical consistency checks.

In [95]:
def process_lotsize(df):
    """Clean and fill lot size with logical consistency"""
    
    # Fix impossible values (lot size < building size)
    impossible_mask = (df['lotsize_sqft'] < df['sqft_num']) & df['lotsize_sqft'].notna() & df['sqft_num'].notna()
    impossible_count = impossible_mask.sum()
    
    if impossible_count > 0:
        df.loc[impossible_mask, 'lotsize_sqft'] = np.nan
        print(f"Set {impossible_count:,} impossible lotsize values to null")
    
    # Fill missing lot sizes using intelligent multipliers
    missing_lotsize = df['lotsize_sqft'].isna().sum()
    if missing_lotsize > 0:
        # Define property-specific multipliers
        multipliers = {
            'single_family': 3.0,
            'townhouse': 2.0,
            'mobile_home': 2.5,
            'multi_family': 1.5,
            'condo_apartment': 1.0,
            'co_op': 1.0,
            'other': 2.0
        }
        
        # Get median lot sizes by property type for land lots
        median_lotsize = df.groupby('propertyCategory')['lotsize_sqft'].median().to_dict()
        
        missing_mask = df['lotsize_sqft'].isna()
        
        # For properties with buildings, use building sqft * multiplier
        for prop_type, multiplier in multipliers.items():
            type_mask = missing_mask & (df['propertyCategory'] == prop_type)
            if type_mask.sum() > 0:
                df.loc[type_mask, 'lotsize_sqft'] = df.loc[type_mask, 'sqft_num'] * multiplier
        
        # For land lots, use median
        land_lot_mask = missing_mask & (df['propertyCategory'] == 'land_lot')
        if land_lot_mask.sum() > 0:
            df.loc[land_lot_mask, 'lotsize_sqft'] = median_lotsize.get('land_lot', 10000)
        
        print(f"Filled {missing_lotsize:,} missing lotsize values")
    
    return df

data = process_lotsize(data)

Set 14,424 impossible lotsize values to null
Filled 70,561 missing lotsize values


## Final Data Quality Check

In [96]:
# Final Data Quality Check
print("Final Data Quality Report")
print("=" * 50)
print(f"Dataset shape: {data.shape}")

# Check for missing values across entire dataset
total_missing = data.isnull().sum().sum()
print(f"Total missing values: {total_missing:,}")

if total_missing > 0:
    print("❌ ERROR: Missing values found!")
    missing_cols = data.isnull().sum()
    for col, count in missing_cols[missing_cols > 0].items():
        print(f"  {col}: {count:,} missing")
else:
    print("✅ SUCCESS: No missing values")

# Quick consistency checks
impossible_lots = ((data['lotsize_sqft'] < data['sqft_num']) & 
                  data['lotsize_sqft'].notna() & data['sqft_num'].notna()).sum()
extreme_beds = (data['beds_num'] > 15).sum()
extreme_baths = (data['baths_num'] > 15).sum()

print(f"\nConsistency checks:")
print(f"Impossible lot sizes: {impossible_lots:,}")
print(f"Extreme beds (>15): {extreme_beds:,}")
print(f"Extreme baths (>15): {extreme_baths:,}")

# Final status
if total_missing == 0 and impossible_lots == 0 and extreme_beds == 0 and extreme_baths == 0:
    print(f"\n✅ Dataset ready for modeling ({len(data):,} records)")
else:
    print(f"\n❌ Issues found - fix before modeling")

Final Data Quality Report
Dataset shape: (269746, 33)
Total missing values: 0
✅ SUCCESS: No missing values

Consistency checks:
Impossible lot sizes: 0
Extreme beds (>15): 0
Extreme baths (>15): 0

✅ Dataset ready for modeling (269,746 records)


## Save Cleaned Dataset

In [97]:
# Save the cleaned dataset
# Remove columns that were only used for validation/cleaning
data = data.drop(columns=['propertyType', 'ppsf'], errors='ignore')

# Standardize column names - remove '_num' suffixes and make consistent
column_mapping = {
    'beds_num': 'beds',
    'baths_num': 'baths', 
    'sqft_num': 'sqft',
    'stories_num': 'stories',
    'lotsize_sqft': 'lot_size',
    'propertyCategory': 'property_category',
    'privatePool': 'private_pool',
    'fireplace_count': 'fireplaces',
    'fireplace_present': 'has_fireplace',
    'cooling_energy_source': 'cooling_energy',
    'heating_energy_source': 'heating_energy',
    'remodeled_year': 'year_remodeled',
    'years_build_to_remodel': 'years_to_remodel'
}

# Apply column renaming
existing_columns = {old: new for old, new in column_mapping.items() if old in data.columns}
data = data.rename(columns=existing_columns)

print(f"Standardized {len(existing_columns)} column names")
print("Column name changes:")
for old, new in existing_columns.items():
    print(f"  {old} → {new}")

data.to_csv('final_cleaned_housing_data.csv', index=False)
print(f"\nSaved cleaned dataset to 'final_cleaned_housing_data.csv'")
print(f"Dataset shape: {data.shape}")
print(f"Final columns: {list(data.columns)}")

Standardized 13 column names
Column name changes:
  beds_num → beds
  baths_num → baths
  sqft_num → sqft
  stories_num → stories
  lotsize_sqft → lot_size
  propertyCategory → property_category
  privatePool → private_pool
  fireplace_count → fireplaces
  fireplace_present → has_fireplace
  cooling_energy_source → cooling_energy
  heating_energy_source → heating_energy
  remodeled_year → year_remodeled
  years_build_to_remodel → years_to_remodel

Saved cleaned dataset to 'final_cleaned_housing_data.csv'
Dataset shape: (269746, 31)
Final columns: ['status', 'street', 'schools', 'zipcode', 'state', 'target', 'private_pool', 'property_category', 'baths', 'beds', 'sqft', 'fireplaces', 'fireplace_location', 'fireplace_type', 'has_fireplace', 'stories', 'cooling_type', 'cooling_energy', 'heating_type', 'heating_energy', 'parking_type', 'parking_spaces', 'has_garage', 'year_built', 'year_remodeled', 'lot_size', 'missing_year_built', 'was_remodeled', 'property_age', 'years_since_remodel', 'ye